# **Modelling Data (Klasifikasi Time Series - Dataset BME)**

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import joblib

df_train = pd.read_csv('BME_TRAIN.csv')
df_test = pd.read_csv('BME_TEST.csv')

X_train = df_train.drop('target', axis=1)
y_train = df_train['target']
X_test = df_test.drop('target', axis=1)
y_test = df_test['target']

def handle_outliers_winsorization(data):
    data_copy = data.copy()
    
    Q1 = data_copy.quantile(0.25)
    Q3 = data_copy.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Capping: nilai di bawah lower_bound diganti dengan lower_bound
    #          nilai di atas upper_bound diganti dengan upper_bound
    for col in data_copy.columns:
        data_copy[col] = data_copy[col].clip(lower=lower_bound[col], upper=upper_bound[col])
    
    return data_copy

X_train = handle_outliers_winsorization(X_train)
X_test = handle_outliers_winsorization(X_test)

scaler = joblib.load('scaler_bme.pkl')

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

X_train_scaled shape: (30, 128)
X_test_scaled shape: (8, 128)


### 4.1 Baseline Model Random Forest

In [2]:
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42)
rf_baseline.fit(X_train_scaled, y_train)

# Evaluasi pada training set
y_pred_train = rf_baseline.predict(X_train_scaled)
acc_train = accuracy_score(y_train, y_pred_train)

print("=" * 60)
print("EVALUASI MODEL RANDOM FOREST - TRAINING SET")
print("=" * 60)
print(f"Akurasi Training Set: {acc_train:.4f} ({acc_train*100:.2f}%)")

print("\nClassification Report (Training Set):")
print(classification_report(y_train, y_pred_train))

EVALUASI MODEL RANDOM FOREST - TRAINING SET
Akurasi Training Set: 1.0000 (100.00%)

Classification Report (Training Set):
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        10
           2       1.00      1.00      1.00        10
           3       1.00      1.00      1.00        10

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



### 4.2 Evaluasi Model pada Test Set

In [3]:
y_pred_test = rf_baseline.predict(X_test_scaled)
acc_test = accuracy_score(y_test, y_pred_test)

print("=" * 60)
print("EVALUASI MODEL RANDOM FOREST - TEST SET")
print("=" * 60)
print(f"Akurasi Test Set: {acc_test:.4f} ({acc_test*100:.2f}%)")

print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred_test))

joblib.dump(rf_baseline, 'model_rf_bme.pkl')

EVALUASI MODEL RANDOM FOREST - TEST SET
Akurasi Test Set: 1.0000 (100.00%)

Classification Report (Test Set):
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         3
           2       1.00      1.00      1.00         3
           3       1.00      1.00      1.00         2

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8



['model_rf_bme.pkl']